In [17]:
%load_ext autotime


time: 154 µs (started: 2026-09-05 19:46:06 +05:30)


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "distilgpt2"  # small causal language model from Hugging Face

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

prompt = "Apple is delicious"
inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=30,    # how long a continuation
        do_sample=True,       # sample (don't loop greedily)
        temperature=0.7,      # how random
        pad_token_id=tokenizer.eos_token_id,
    )

generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

print("Prompt:")
print(prompt)
print("\nGenerated continuation:")
print(generated_text)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Prompt:
Apple is delicious

Generated continuation:
Apple is delicious and easy to make. It's easy to make when you are done with it. However, it has a few drawbacks.





time: 3.45 s (started: 2026-09-05 19:46:06 +05:30)


# TinyLlama LoRA Fine-Tuning on Apple Silicon (M-series)

## Goal

Take `TinyLlama/TinyLlama-1.1B-Chat-v1.0` and adapt it into a customer-support agent for a fictional brand called **TechMart Electronics** using **LoRA**.

The notebook demonstrates the full practical loop:

1. Load a chat-tuned LLM.
2. Format prompts using the model's chat template.
3. Build a small supervised fine-tuning dataset.
4. Compute loss only on the assistant response, not on the user/system prompt.
5. Attach LoRA adapters to the model.
6. Train only the LoRA weights.
7. Compare base-model vs LoRA-adapted replies.
8. Save the tiny adapter file.

---

## What kind of fine-tuning is this?

This is best described as:

```text
chat-style supervised fine-tuning using LoRA and response-only loss masking
```

It is supervised fine-tuning because every training example has:

```text
input conversation -> ideal assistant response
```

It is chat/instruction fine-tuning because the input is formatted as:

```text
system instruction + user message -> assistant response
```

It uses LoRA because we do **not** update the full TinyLlama model. We freeze the base model and train only small low-rank adapter matrices.

## Why TinyLlama?

`TinyLlama/TinyLlama-1.1B-Chat-v1.0` is useful for a teaching notebook because:

- It is small enough to run on a Mac with Apple Silicon.
- It already speaks coherent English.
- It uses a Llama-style transformer architecture, so the ideas transfer to Llama, Mistral, Qwen-style models, etc.
- It is already chat-tuned, so it understands roles like `system`, `user`, and `assistant`.

We are **not** teaching the model English from scratch. We are nudging its behavior toward a TechMart support-agent style.

## Why LoRA?

A normal transformer layer has large weight matrices, for example `W`.

Full fine-tuning would directly update `W`.

LoRA freezes `W` and learns a small low-rank update:

```text
W_effective = W + scaling * low_rank_update
```

where:

```text
low_rank_update = B @ A
scaling = lora_alpha / r
```

In this notebook:

```text
r = 8
lora_alpha = 16
scaling = 16 / 8 = 2
```

So conceptually:

```text
W_effective = W + 2 * (B @ A)
```

In practice, PEFT usually keeps the base weight and LoRA weights separate during training. It behaves as if the update were added, without permanently modifying the original base weights. You can later merge them if needed.

## Required packages

Install these first:

```bash
pip install torch transformers datasets peft accelerate sentencepiece safetensors
```

The first run will download TinyLlama into your Hugging Face cache. The model is roughly 2.2 GB in bf16/fp16 form.

In [19]:
# ── 1. Environment check ─────────────────────────────────────────────────
# This cell verifies:
#   1. Required packages are installed.
#   2. PyTorch can see Apple Silicon GPU via MPS, or CUDA, or CPU.
#   3. We choose a reasonable compute dtype.

import importlib.util
import warnings

warnings.filterwarnings("ignore")

required = ["torch", "transformers", "peft", "datasets", "accelerate"]
missing = [p for p in required if importlib.util.find_spec(p) is None]
if missing:
    raise RuntimeError(f"Missing packages: {missing}. Install them first.")

import torch

# Pick the best available device.
# MPS = Apple Metal Performance Shaders backend.
if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

# Prefer bfloat16 where practical. On some older MPS/PyTorch/macOS combinations,
# bfloat16 may be less reliable than float16. The small smoke test below picks a
# dtype that should work on your machine.
def pick_dtype(device: str):
    if device == "cuda":
        # Most modern NVIDIA GPUs used for LLM work support bf16.
        if torch.cuda.is_bf16_supported():
            return torch.bfloat16
        return torch.float16

    if device == "mps":
        # MPS support varies by PyTorch/macOS version. Try bf16, fall back to fp16.
        try:
            _ = torch.ones(1, device="mps", dtype=torch.bfloat16) + 1
            return torch.bfloat16
        except Exception:
            return torch.float16

    # CPU can use fp32 reliably. Training will be slow, but it avoids dtype surprises.
    return torch.float32

DTYPE = pick_dtype(DEVICE)

print(f"PyTorch      : {torch.__version__}")
print(f"Device       : {DEVICE}")
print(f"Compute dtype: {DTYPE}")

PyTorch      : 2.10.0
Device       : mps
Compute dtype: torch.bfloat16
time: 1.82 ms (started: 2026-09-05 19:46:10 +05:30)


## 2. Load TinyLlama-Chat

We use the **chat-tuned** checkpoint:

```text
TinyLlama/TinyLlama-1.1B-Chat-v1.0
```

This matters because a chat model expects prompts in a specific conversational format.

A Python list like this:

```python
[
    {"role": "system", "content": "You are helpful."},
    {"role": "user", "content": "How do I reset my password?"},
]
```

is **not** what the model directly receives. The model receives token IDs. Before tokenization, the messages must be converted into a single prompt string with the model's expected special tokens.

That is exactly what `tokenizer.apply_chat_template(...)` does.

In [20]:
# ── 2. Load tokenizer and base model ─────────────────────────────────────

from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# The tokenizer converts text <-> token IDs.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Llama-family tokenizers often do not define a dedicated pad token.
# For batched training, all examples must have equal length, so we need padding.
# A common convention is to reuse EOS as PAD.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model weights directly in the chosen dtype.
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=DTYPE,
).to(DEVICE)

# KV cache is useful for generation/inference, not training.
# We enable it now for baseline generation and disable it later for training.
base_model.config.use_cache = True

# TinyLlama ships a built-in max_length=2048. Clearing it lets our max_new_tokens
# take over cleanly (otherwise generate() prints a noisy "both were set" warning).
base_model.generation_config.max_length = None

total_params = sum(p.numel() for p in base_model.parameters())
print(f"Loaded: {MODEL_NAME}")
print(f"Total parameters: {total_params:,} ({total_params / 1e6:.1f}M)")
print(f"Approx weight memory at current dtype: {total_params * torch.tensor([], dtype=DTYPE).element_size() / 1e9:.2f} GB")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loaded: TinyLlama/TinyLlama-1.1B-Chat-v1.0
Total parameters: 1,100,048,384 (1100.0M)
Approx weight memory at current dtype: 2.20 GB
time: 3.73 s (started: 2026-09-05 19:46:10 +05:30)


## How do we know whether to use `apply_chat_template`?

Use it when you are working with a **chat** or **instruct** model.

Model names often contain words like:

```text
Chat
Instruct
Assistant
SFT
RLHF
DPO
```

Different chat models use different special-token formats. For example, one model may expect:

```text
<|user|>
...
<|assistant|>
```

while another may expect:

```text
[INST] ... [/INST]
```

So do **not** manually invent the prompt format unless you have to. First inspect the tokenizer's chat template.

In [21]:
# ── Inspect the model's chat template ─────────────────────────────────────
# This is useful when learning a new chat model.
# If this prints a Jinja-style template, the model/tokenizer knows how to format chat messages.

print(tokenizer.chat_template[:1000] if tokenizer.chat_template else "No chat template found.")

{% for message in messages %}
{% if message['role'] == 'user' %}
{{ '<|user|>
' + message['content'] + eos_token }}
{% elif message['role'] == 'system' %}
{{ '<|system|>
' + message['content'] + eos_token }}
{% elif message['role'] == 'assistant' %}
{{ '<|assistant|>
'  + message['content'] + eos_token }}
{% endif %}
{% if loop.last and add_generation_prompt %}
{{ '<|assistant|>' }}
{% endif %}
{% endfor %}
time: 248 µs (started: 2026-09-05 19:46:13 +05:30)


## 3. Baseline generation before fine-tuning

Before training anything, we test the untouched model.

This gives us a reference point:

```text
base model reply -> fine-tuned model reply
```

If we skip this, we cannot tell whether LoRA training actually changed anything.

In [22]:
# ── 3. Baseline generation helper ────────────────────────────────────────

SYSTEM_PROMPT = (
    "You are a friendly, concise customer support agent for TechMart "
    "Electronics. Acknowledge the customer's frustration, give a clear next "
    "step, and keep replies under three sentences."
)


def generate_reply(model, user_message, system_prompt=SYSTEM_PROMPT):
    """Generate one assistant reply from a model.

    Key ideas:
    - We build a list of chat messages.
    - `apply_chat_template` converts those messages into the exact text format
      expected by this chat model.
    - `add_generation_prompt=True` appends the assistant-turn marker, telling
      the model: "now generate the assistant response".
    - `generate()` returns prompt tokens + generated tokens, so we slice off the
      original prompt and decode only the newly generated tokens.
    """

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message},
    ]

    # Step 1: format chat messages into a single prompt string.
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # Step 2: tokenize the prompt string into tensor IDs.
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

    model.eval()
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=True,
            temperature=0.7,
            repetition_penalty=1.1,
        )

    # Shape explanation:
    # output_ids has shape [batch_size, total_sequence_length].
    # Here batch_size is 1, so output_ids[0] selects the only generated sequence.
    #
    # `inputs["input_ids"].shape[1]` is the number of prompt tokens.
    # `output_ids[0, prompt_len:]` removes the prompt and keeps only new tokens.
    prompt_len = inputs["input_ids"].shape[1]
    new_tokens = output_ids[0, prompt_len:]

    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

time: 615 µs (started: 2026-09-05 19:46:13 +05:30)


## Why `output_ids[0, prompt_len:]`?

`model.generate(...)` returns a 2D tensor:

```text
[batch_size, total_sequence_length]
```

For one prompt, the shape may be:

```text
torch.Size([1, 105])
```

The `0` selects the first and only generated sequence:

```python
output_ids[0]
```

The `prompt_len:` slice removes the original prompt tokens and keeps only the newly generated assistant tokens:

```python
output_ids[0, prompt_len:]
```

So this line:

```python
new_tokens = output_ids[0, prompt_len:]
```

means:

```text
From the only generated sequence, keep only the generated reply, not the input prompt.
```

## What is KV cache and why can it be reused?

During generation, the model produces one token at a time:

```text
Step 1: "I'm"
Step 2: "sorry"
Step 3: "for"
Step 4: "the"
...
```

At each step, the new token attends to all previous tokens.

Without KV cache, the model repeatedly recomputes key/value tensors for the same old prompt tokens.

With KV cache:

```text
old tokens -> reuse cached keys/values
new token  -> compute fresh query/key/value
```

Important distinction:

```text
KV cache is not reused across unrelated user inputs.
```

Each input/request gets its own cache. The reuse happens only inside the same generation call while producing the continuation.

This is why `use_cache=True` is useful during inference/generation.

During training, we usually set `use_cache=False` because training processes full sequences with gradients and the cache is not needed.

In [23]:
# ── Run baseline prompts before fine-tuning ──────────────────────────────

test_prompts = [
    "My order #4521 hasn't arrived after 2 weeks.",
    "I want a refund for my broken headphones.",
    "Your app keeps crashing on my phone.",
]

print("=" * 80)
print("  BASE MODEL — replies before fine-tuning")
print("=" * 80)

for q in test_prompts:
    print(f"\nCustomer: {q}")
    print(f"Agent   : {generate_reply(base_model, q)}")

  BASE MODEL — replies before fine-tuning

Customer: My order #4521 hasn't arrived after 2 weeks.
Agent   : 

Customer: I want a refund for my broken headphones.
Agent   : 

Customer: Your app keeps crashing on my phone.
Agent   : 
time: 14.8 s (started: 2026-09-05 19:46:13 +05:30)


## 4. Build the training set

Each example is:

```text
customer message -> ideal support-agent reply
```

The replies intentionally follow a pattern:

1. Acknowledge or apologize.
2. Give one concrete next step.
3. Stay short.

With only 15 examples, this is **not** production fine-tuning. It is enough to demonstrate the mechanism and shift the model's surface style.

For a real support bot, you would want hundreds or thousands of curated examples plus a held-out evaluation set.

In [24]:
# ── 4. Small supervised chat dataset ─────────────────────────────────────
# Each tuple is: (customer_message, ideal_assistant_reply)


support_examples = [
    ("My order hasn't arrived yet.",
     "Arr, sorry for the delay, matey — that be vexin'. Could ye share yer order number so I can pull up the trackin' right now?"),
    ("I want a refund for my broken headphones.",
     "That not be the voyage we want for ye. I be startin' a full refund now — ye'll see the doubloons back on yer card in 3–5 business days."),
    ("How do I reset my password?",
     "Happy to help, matey. Open Settings → Account → Reset Password, and we'll send ye a secure reset link within a minute."),
    ("Your app keeps crashing on my phone.",
     "Sorry for the squall. Update to the latest app version and clear the cache; if she still be crashin', send me yer phone model and I'll escalate it up the mast."),
    ("I was charged twice for the same item.",
     "Apologies — that be plain wrong. I spy the duplicate charge and I be reversin' it now; the doubloons will land back within 5 business days."),
    ("Can I change my delivery address?",
     "Aye, as long as the order hasn't set sail. Send me the new address and I'll update it on the spot."),
    ("The product I received is the wrong color.",
     "Sorry for the mix-up, matey! I be shippin' the right color today at no charge — keep the wrong one, no need to send it back."),
    ("I need to cancel my subscription.",
     "No worries, I be cancellin' it now. Yer access stays afloat until the end of the current billing period."),
    ("The website won't accept my coupon code.",
     "Let's set that right. Codes be case-sensitive and some expire — could ye paste the exact code so I can check it on me side?"),
    ("I never received my confirmation email.",
     "Sorry 'bout that, matey. Confirm the email on yer account and I'll resend it right away — also worth checkin' yer spam locker."),
    ("My package arrived damaged.",
     "That be a cryin' shame — I be sendin' a replacement at no charge today. Ye can keep or recycle the damaged one, no return needed."),
    ("How long does shipping usually take?",
     "Standard shippin' be 5–7 business days. We also offer express (2–3 days) and overnight if ye need it sooner, matey."),
    ("Do you ship internationally?",
     "Aye, we sail to 40+ countries. Add an item to yer cart and the international rates will appear at checkout."),
    ("My order shows delivered but I never got it.",
     "That be stressful, matey — I be filin' a lost-package claim now and shippin' a replacement today. Could ye confirm the delivery address on file?"),
    ("Is the warranty transferable if I gift this?",
     "Aye — the one-year warranty covers the device, not the buyer, so the lucky recipient be fully covered."),
]

print(f"Dataset size: {len(support_examples)} examples")

Dataset size: 15 examples
time: 734 µs (started: 2026-09-05 19:46:28 +05:30)


In [25]:
system_prompt = SYSTEM_PROMPT


pairs = support_examples[:3]  # just a few examples to keep it simple
for user_msg, assistant_msg in pairs:
    # ── Step A: build the *prefix* the model would see at inference ──
    # This is system + user + the "<|assistant|>\n" header. We need its
    # length (in tokens) so we know where to start computing loss.
    prefix_messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_msg},
    ]
    prefix_text = tokenizer.apply_chat_template(
        prefix_messages,
        tokenize=False,
        add_generation_prompt=True,  # appends the assistant turn opener
    )

    # ── Step B: build the *full* text including the assistant reply ──
    full_messages = prefix_messages + [
        {"role": "assistant", "content": assistant_msg},
    ]
    full_text = tokenizer.apply_chat_template(
        full_messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    break

print("Prefix (model input at inference):")
print(prefix_text)
print("\nFull text (model input during training):")
print(full_text)


Prefix (model input at inference):
<|system|>
You are a friendly, concise customer support agent for TechMart Electronics. Acknowledge the customer's frustration, give a clear next step, and keep replies under three sentences.</s>
<|user|>
My order hasn't arrived yet.</s>
<|assistant|>


Full text (model input during training):
<|system|>
You are a friendly, concise customer support agent for TechMart Electronics. Acknowledge the customer's frustration, give a clear next step, and keep replies under three sentences.</s>
<|user|>
My order hasn't arrived yet.</s>
<|assistant|>
Arr, sorry for the delay, matey — that be vexin'. Could ye share yer order number so I can pull up the trackin' right now?</s>

time: 642 µs (started: 2026-09-05 19:46:28 +05:30)


## 5. Tokenize with response-only loss masking

This is the most important training detail.

For a chat example like:

```text
System: You are a TechMart support agent.
User: My order is late.
Assistant: I'm sorry for the delay. Could you share your order number?
```

we do **not** want the model to learn to predict the system prompt or user question.

We only want the model to learn the assistant answer.

So the loss should apply only to:

```text
I'm sorry for the delay. Could you share your order number?
```

not to:

```text
System: ...
User: ...
Assistant header
```

In Hugging Face/PyTorch language-model training, label value `-100` means:

```text
ignore this token position when computing loss
```

So we set:

```text
labels = input_ids for assistant tokens
labels = -100 for system/user/header/padding tokens
```

### Quick detour: two different masks, on a tiny toy example

The dataset code below packs a lot into one loop, so let's first isolate the *idea* on a
hand-made 11-token sequence — **no model, no tokenizer**. There are **two separate masks**
doing **two separate jobs**, and they are easy to mix up:

| mask | question it answers | rule |
|---|---|---|
| **attention mask** | *which tokens may the model look at?* | `0` on padding → those positions get **zero attention** |
| **labels (`-100`)** | *which tokens is the model graded on?* | `-100` on system + user + padding → **loss ignores them**; only the assistant reply is learned |

The **padding** is ignored by *both*. The **system + user prompt** is genuine input (the model must
read it), so it keeps `attention = 1` — but we still **don't want to train the model to predict it**,
so its label is `-100`. Only the **assistant reply** gets a real label.

In [26]:
# ── Mask #1: the ATTENTION mask — padding must not soak up attention ──────
import torch

# Imagine ONE query token scoring how relevant each of 5 key positions is.
# The last two positions are PADDING (only there to make the batch rectangular),
# and — awkwardly — they happen to have high raw scores.
scores    = torch.tensor([2.0, 1.0, 0.5, 3.0, 2.5])   # raw attention scores (higher = more relevant)
attn_mask = torch.tensor([1,   1,   1,   0,   0])      # 1 = real token, 0 = padding

def rounded(t):
    return [round(v, 3) for v in t.tolist()]

# WITHOUT masking, softmax happily hands most of the attention to the padding slots (wrong!).
print("attention if we IGNORE the mask :", rounded(torch.softmax(scores, dim=-1)))

# WITH masking, we set padding scores to -inf *before* softmax, so they get exactly 0 weight.
masked = scores.masked_fill(attn_mask == 0, float("-inf"))
print("attention USING the mask        :", rounded(torch.softmax(masked, dim=-1)))

print("\n-> without the mask, 73% of attention leaks onto the 2 padding tokens;")
print("   with the mask, padding gets 0.0 and the real tokens share 100%.")

attention if we IGNORE the mask : [0.168, 0.062, 0.037, 0.456, 0.277]
attention USING the mask        : [0.629, 0.231, 0.14, 0.0, 0.0]

-> without the mask, 73% of attention leaks onto the 2 padding tokens;
   with the mask, padding gets 0.0 and the real tokens share 100%.
time: 933 µs (started: 2026-09-05 19:46:28 +05:30)


In [27]:
# ── Mask #2: the LABEL mask — only the assistant reply counts toward loss ──
# A tiny "already tokenized" training example (tokens shown as words so we can read them).
tokens  = ["<sys>", "be", "kind", "<user>", "help", "<asst>", "sure", "!", "</s>", "<pad>", "<pad>"]
section = ["prompt","prompt","prompt","prompt","prompt","prompt","reply","reply","reply","pad","pad"]

IGNORE = -100  # PyTorch's cross-entropy skips any position whose label == -100

# attention: 1 for real tokens (prompt + reply), 0 for padding
attention_mask = [0 if s == "pad" else 1 for s in section]

# labels: the token itself ONLY where it's the assistant reply; -100 everywhere else.
# This is exactly what the real dataset does with:
#     labels[:prefix_len]          = -100   # the prompt  (system + user + header)
#     labels[attention_mask == 0]  = -100   # the padding
labels = [tok if s == "reply" else IGNORE for tok, s in zip(tokens, section)]

print(f"{'pos':<4}{'token':<9}{'section':<9}{'attention':<11}{'label (what loss learns)'}")
print("-" * 56)
for i, (tok, s, a, lab) in enumerate(zip(tokens, section, attention_mask, labels)):
    shown = lab if lab != IGNORE else "-100  (ignored)"
    print(f"{i:<4}{tok:<9}{s:<9}{a:<11}{shown}")

learned = sum(1 for lab in labels if lab != IGNORE)
print(f"\n-> loss is computed on only {learned} of {len(tokens)} positions (the assistant reply):")
print("   system + user prompt : read, but NOT graded  (attention 1, label -100)")
print("   padding              : neither read nor graded (attention 0, label -100)")

pos token    section  attention  label (what loss learns)
--------------------------------------------------------
0   <sys>    prompt   1          -100  (ignored)
1   be       prompt   1          -100  (ignored)
2   kind     prompt   1          -100  (ignored)
3   <user>   prompt   1          -100  (ignored)
4   help     prompt   1          -100  (ignored)
5   <asst>   prompt   1          -100  (ignored)
6   sure     reply    1          sure
7   !        reply    1          !
8   </s>     reply    1          </s>
9   <pad>    pad      0          -100  (ignored)
10  <pad>    pad      0          -100  (ignored)

-> loss is computed on only 3 of 11 positions (the assistant reply):
   system + user prompt : read, but NOT graded  (attention 1, label -100)
   padding              : neither read nor graded (attention 0, label -100)
time: 846 µs (started: 2026-09-05 19:46:28 +05:30)


In [28]:
# ── 5. Dataset with response-only masking ────────────────────────────────

from torch.utils.data import Dataset, DataLoader

MAX_LENGTH = 256


class SupportChatDataset(Dataset):
    """Convert (user, assistant) pairs into tensors for chat SFT.

    For every example we produce:
    - input_ids: full chat sequence tokens
    - attention_mask: 1 for real tokens, 0 for padding
    - labels: same as input_ids only for assistant-response tokens;
              -100 everywhere else

    The model sees the full conversation, but loss is computed only on the
    assistant reply.
    """

    def __init__(self, pairs, tokenizer, system_prompt, max_length=MAX_LENGTH):
        self.items = []

        for user_msg, assistant_msg in pairs:
            # A. Build the prefix used at inference time:
            #    system + user + assistant header.
            #
            # We need this prefix length so we know where the assistant answer
            # begins in the full training sequence.
            prefix_messages = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_msg},
            ]
            prefix_text = tokenizer.apply_chat_template(
                prefix_messages,
                tokenize=False,
                add_generation_prompt=True,
            )

            # B. Build the full training text:
            #    system + user + assistant answer.
            full_messages = prefix_messages + [
                {"role": "assistant", "content": assistant_msg},
            ]
            full_text = tokenizer.apply_chat_template(
                full_messages,
                tokenize=False,
                add_generation_prompt=False,
            )

            # C. Tokenize full text with fixed padding.
            #    This makes every item exactly max_length tokens.
            full_enc = tokenizer(
                full_text,
                truncation=True,
                max_length=max_length,
                padding="max_length",
                return_tensors="pt",
            )

            # D. Tokenize prefix separately to find the assistant-answer boundary.
            #    `add_special_tokens=False` is important because the chat template
            #    has already inserted special tokens.
            prefix_enc = tokenizer(
                prefix_text,
                truncation=True,
                max_length=max_length,
                add_special_tokens=False,
                return_tensors="pt",
            )

            # Remove the batch dimension from each single example.
            # Before squeeze: [1, max_length]
            # After squeeze : [max_length]
            input_ids = full_enc["input_ids"].squeeze(0)
            attention_mask = full_enc["attention_mask"].squeeze(0)
            prefix_len = prefix_enc["input_ids"].shape[1]

            # E. Create labels.
            #    Start from a copy of input_ids, then mask everything except the
            #    assistant reply.
            labels = input_ids.clone()
            labels[:prefix_len] = -100          # ignore system + user + assistant header
            labels[attention_mask == 0] = -100  # ignore padding

            self.items.append(
                {
                    "input_ids": input_ids,
                    "attention_mask": attention_mask,
                    "labels": labels,
                }
            )

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        return self.items[idx]


dataset = SupportChatDataset(support_examples, tokenizer, SYSTEM_PROMPT)
loader = DataLoader(dataset, batch_size=2, shuffle=True)

# Sanity check: the number of trainable-label tokens should roughly match the
# assistant reply length, not the whole sequence length.
first = dataset[0]
n_train = (first["labels"] != -100).sum().item()
n_total = first["attention_mask"].sum().item()
print(f"Example 0: {n_total} non-pad tokens, {n_train} of them contribute to loss")
print(f"           ({n_train / n_total:.0%} of visible tokens contribute to loss)")

Example 0: 110 non-pad tokens, 38 of them contribute to loss
           (35% of visible tokens contribute to loss)
time: 8.35 ms (started: 2026-09-05 19:46:28 +05:30)


In [29]:
items = []
pairs = support_examples[:3]  # just a few examples to keep it simple
for user_msg, assistant_msg in pairs:
    # ── Step A: build the *prefix* the model would see at inference ──
    # This is system + user + the "<|assistant|>\n" header. We need its
    # length (in tokens) so we know where to start computing loss.
    prefix_messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_msg},
    ]
    prefix_text = tokenizer.apply_chat_template(
        prefix_messages,
        tokenize=False,
        add_generation_prompt=True,  # appends the assistant turn opener
    )

    # ── Step B: build the *full* text including the assistant reply ──
    full_messages = prefix_messages + [
        {"role": "assistant", "content": assistant_msg},
    ]
    full_text = tokenizer.apply_chat_template(
        full_messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    # ── Step C: tokenize both ────────────────────────────────────────
    # We tokenize the full text with padding so all examples in a
    # ── Step C: tokenize both ────────────────────────────────────────
            # We tokenize the full text with padding so all examples in a
            # batch are the same length. We tokenize the prefix without any
            # padding/special-token funny business so its length is exact.
    max_length = 256
    full_enc = tokenizer(
        full_text,
        truncation=True,
        max_length=max_length,
        padding="max_length",
        return_tensors="pt",
    )
    prefix_enc = tokenizer(
        prefix_text,
        truncation=True,
        max_length=max_length,
        add_special_tokens=False,  # apply_chat_template already added them
        return_tensors="pt",
    )

    input_ids      = full_enc["input_ids"].squeeze(0) # batch-size x seq-length
    attention_mask = full_enc["attention_mask"].squeeze(0) # I don't want to pay any attention to padded tokens, i.e. EOS tokens
    prefix_len     = prefix_enc["input_ids"].shape[1]

    # ── Step D: build labels with response-only masking ──────────────
    # Start from a copy of input_ids, then zap everything we don't
    # want the loss to see.
    labels = input_ids.clone()
    labels[:prefix_len] = -100               # ignore system + user + header
    labels[attention_mask == 0] = -100       # ignore padding

    items.append({
        "input_ids":      input_ids,
        "attention_mask": attention_mask,
        "labels":         labels,
    })
    break

print(items[0])

{'input_ids': tensor([    1,   529, 29989,  5205, 29989, 29958,    13,  3492,   526,   263,
        19780, 29892,  3022,   895, 11962,  2304, 10823,   363,  1920,   305,
        15838, 28251,  1199, 29889,   319, 15415,  5485,   278, 11962, 29915,
        29879,  1424, 11036, 29892,  2367,   263,  2821,  2446,  4331, 29892,
          322,  3013,  1634,  3687,  1090,  2211, 25260, 29889,     2,    13,
        29966, 29989,  1792, 29989, 29958,    13,  3421,  1797, 22602, 29915,
        29873, 11977,  3447, 29889,     2,    13, 29966, 29989,   465, 22137,
        29989, 29958,    13, 16401, 29892,  7423,   363,   278,  9055, 29892,
        15358, 29891,   813,   393,   367,   325,   735,   262,  4286,  6527,
         8007,  6232,   343,   261,  1797,  1353,   577,   306,   508,  8206,
          701,   278,  5702,   262, 29915,  1492,  1286, 29973,     2,    13,
            2,     2,     2,     2,     2,     2,     2,     2,     2,     2,
            2,     2,     2,     2,     2,     2, 

## 6. Attach LoRA adapters

We now wrap the base model with LoRA.

The original TinyLlama weights remain frozen. Only the small LoRA matrices are trainable.

For Llama-style attention layers, common target modules are:

```text
q_proj: query projection
k_proj: key projection
v_proj: value projection
o_proj: attention output projection
```

These are central to attention behavior, so they are good first targets for LoRA.

You can also target MLP modules like:

```text
gate_proj, up_proj, down_proj
```

for more capacity, but this notebook keeps the demo lightweight.

In [30]:
# ── 6. Configure and attach LoRA ─────────────────────────────────────────

from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

# This freezes the base model and inserts trainable LoRA adapters into the
# selected target modules.
model = get_peft_model(base_model, lora_config)

# Disable KV cache during training. It is an inference optimization, not needed
# for full-sequence training with gradients.
model.config.use_cache = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print("=" * 60)
print("  LoRA wraps TinyLlama")
print("=" * 60)
print(f"  Total parameters     : {total:>14,}")
print(f"  Trainable LoRA params: {trainable:>14,}")
print(f"  Frozen base params   : {total - trainable:>14,}")
print(f"  % trainable          : {100 * trainable / total:>13.4f}%")

  LoRA wraps TinyLlama
  Total parameters     :  1,102,301,184
  Trainable LoRA params:      2,252,800
  Frozen base params   :  1,100,048,384
  % trainable          :        0.2044%
time: 181 ms (started: 2026-09-05 19:46:28 +05:30)


## LoRA scaling recap

The LoRA forward pass behaves like:

```text
original_output = x @ W
lora_output     = x @ A @ B
final_output    = original_output + (lora_alpha / r) * lora_output
```

In this notebook:

```text
lora_alpha / r = 16 / 8 = 2
```

So the LoRA update is multiplied by 2 before being added to the frozen base-layer output.

The base weight `W` is not normally modified during training. The adapter path is kept separate. If later you call `merge_and_unload()`, the adapter update can be physically merged into the base weights.

## 7. Train

This is a plain PyTorch training loop, not Hugging Face `Trainer`, so every step is visible.

For each batch:

```text
1. Move tensors to device.
2. Run forward pass.
3. Hugging Face computes causal LM loss because labels are provided.
4. Loss ignores positions where labels == -100.
5. Backpropagate.
6. Optimizer updates only LoRA weights.
```

The base model remains frozen throughout.

In [31]:
# ── 7. Train LoRA adapter ────────────────────────────────────────────────

import time
from torch.optim import AdamW

NUM_EPOCHS = 5
LEARNING_RATE = 2e-4

optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
)

model.train()
loss_history = []

print(f"Training on {DEVICE} for {NUM_EPOCHS} epochs...")
print("-" * 60)
start = time.time()

for epoch in range(NUM_EPOCHS):
    epoch_loss = 0.0

    for batch in loader:
        # Move input_ids, attention_mask, labels to the same device as model.
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        optimizer.zero_grad()

        # Because labels are passed, the model returns outputs.loss.
        # That loss is computed only where labels != -100.
        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(loader)
    loss_history.append(avg_loss)

    bar_len = 30
    filled = int(bar_len * (epoch + 1) / NUM_EPOCHS)
    bar = "█" * filled + "░" * (bar_len - filled)
    print(f"  Epoch {epoch + 1}/{NUM_EPOCHS} |{bar}| loss = {avg_loss:.4f}")

elapsed = time.time() - start
print("-" * 60)
print(f"Done in {elapsed:.1f}s ({elapsed / NUM_EPOCHS:.1f}s per epoch)")

Training on mps for 5 epochs...
------------------------------------------------------------
  Epoch 1/5 |██████░░░░░░░░░░░░░░░░░░░░░░░░| loss = 3.6104
  Epoch 2/5 |████████████░░░░░░░░░░░░░░░░░░| loss = 2.5948
  Epoch 3/5 |██████████████████░░░░░░░░░░░░| loss = 2.1685
  Epoch 4/5 |████████████████████████░░░░░░| loss = 1.7236
  Epoch 5/5 |██████████████████████████████| loss = 1.2896
------------------------------------------------------------
Done in 26.3s (5.3s per epoch)
time: 26.3 s (started: 2026-09-05 19:46:29 +05:30)


## 8. Compare base vs fine-tuned replies

The PEFT wrapper lets us disable the adapter temporarily:

```python
with model.disable_adapter():
    base_reply = generate_reply(model, prompt)
```

That gives the base-model response without reloading the model.

Then this:

```python
ft_reply = generate_reply(model, prompt)
```

uses the LoRA adapter.

So the comparison is clean:

```text
same prompt
same generation settings
adapter off -> base model
adapter on  -> fine-tuned model
```

In [32]:
# ── 8. Side-by-side generation ──────────────────────────────────────────

# Re-enable the KV cache for fast generation (training turned it off).
model.config.use_cache = True
base_model.config.use_cache = True

eval_prompts = [
    "My order #4521 hasn't arrived after 2 weeks.",
    "I want a refund.",
    "Your app keeps crashing.",
    "My laptop screen is flickering since the last update.",
]

print("=" * 90)
print("  BASE  vs  FINE-TUNED")
print("=" * 90)

for q in eval_prompts:
    print(f"\nCustomer    : {q}")

    # Adapter disabled: base TinyLlama behavior.
    with model.disable_adapter():
        base_reply = generate_reply(model, q)

    # Adapter enabled: TechMart LoRA behavior.
    ft_reply = generate_reply(model, q)

    print(f"  base      : {base_reply}")
    print(f"  fine-tuned: {ft_reply}")

  BASE  vs  FINE-TUNED

Customer    : My order #4521 hasn't arrived after 2 weeks.
  base      : 
  fine-tuned: 

Customer    : I want a refund.
  base      : 
  fine-tuned: 

Customer    : Your app keeps crashing.
  base      : 
  fine-tuned: 

Customer    : My laptop screen is flickering since the last update.
  base      : 
  fine-tuned: 
time: 24.7 s (started: 2026-09-05 19:46:55 +05:30)


## 9. Save the LoRA adapter

`model.save_pretrained(...)` saves only the LoRA adapter weights and config, not the full base model.

That is the deployment benefit:

```text
Base model: large, downloaded once
Adapter: small, task-specific, easy to swap
```

Later you can reload the adapter like this:

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16,
)
model = PeftModel.from_pretrained(base, "./tinyllama-techmart-lora")
```

If you want a single merged model artifact, you can later use:

```python
merged_model = model.merge_and_unload()
```

In [33]:
# ── 9. Save adapter and compare storage size ─────────────────────────────

import os

ADAPTER_DIR = "./tinyllama-techmart-lora"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

adapter_bytes = sum(
    os.path.getsize(os.path.join(ADAPTER_DIR, f))
    for f in os.listdir(ADAPTER_DIR)
    if os.path.isfile(os.path.join(ADAPTER_DIR, f))
)

base_bytes = sum(p.numel() * p.element_size() for p in base_model.parameters())

print("Storage Comparison")
print("=" * 50)
print(f"  Base model weights : {base_bytes / 1024 / 1024:>8.1f} MB")
print(f"  LoRA adapter       : {adapter_bytes / 1024 / 1024:>8.1f} MB")
print(f"  Adapter / base     : {adapter_bytes / base_bytes * 100:>7.3f}%")
print("=" * 50)
print(f"\nAdapter saved to: {os.path.abspath(ADAPTER_DIR)}")

Storage Comparison
  Base model weights :   2106.8 MB
  LoRA adapter       :     12.1 MB
  Adapter / base     :   0.573%

Adapter saved to: /Users/shivam13juna/Documents/scaler/IITR_REF/fine_tuning/tinyllama-techmart-lora
time: 878 ms (started: 2026-09-05 19:47:20 +05:30)


# Wrap-up

What this notebook demonstrated:

| Step | Action | Why it matters |
|---|---|---|
| 1 | Checked environment and device | Makes the notebook portable across MPS/CUDA/CPU |
| 2 | Loaded TinyLlama-Chat | Starts from a real chat-tuned Llama-style model |
| 3 | Used `apply_chat_template` | Formats conversations the way the model expects |
| 4 | Built support examples | Defines the desired TechMart support behavior |
| 5 | Used response-only masking | Trains only on assistant replies, not the prompt |
| 6 | Added LoRA adapters | Updates tiny trainable matrices, freezes base model |
| 7 | Trained with a visible PyTorch loop | Shows exactly what happens during fine-tuning |
| 8 | Compared adapter off vs on | Verifies behavior changed because of LoRA |
| 9 | Saved adapter | Produces the small shippable artifact |

## Key takeaways

1. Chat models need the right prompt format. Use `tokenizer.apply_chat_template(...)`.
2. `generate()` returns prompt + completion, so slice off the prompt with `output_ids[0, prompt_len:]`.
3. KV cache is reused only within a single generation call, not across unrelated inputs.
4. In chat SFT, loss should usually be computed only on assistant tokens.
5. LoRA learns a scaled low-rank update while the original model weights remain frozen.
6. The adapter is small because it stores only LoRA weights, not the full base model.

## Where to go next

For a more realistic project:

- Increase the dataset from 15 examples to hundreds/thousands.
- Add a validation set.
- Track validation loss, not just training loss.
- Add evaluation prompts that test policy compliance and edge cases.
- Consider targeting MLP modules too: `gate_proj`, `up_proj`, `down_proj`.
- Try `r=16` or `r=32` if the model does not adapt strongly enough.
- Use deterministic decoding during evaluation if you want stable comparisons:

```python
do_sample=False
```